In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from collections import  Counter
import SimpleITK as sitk
import nibabel as nib
import time
from tqdm import tqdm

In [2]:
def sitk_new_blank_image(size, spacing, direction, origin, default_value=0.):
    image = sitk.GetImageFromArray(np.ones(size, dtype=np.float).T * default_value)
    image.SetSpacing(spacing)
    image.SetDirection(direction)
    image.SetOrigin(origin)
    return image

def sitk_resample_to_image(image, reference_image, interpolator, default_value=0., transform=None,
                           output_pixel_type=None):
    if transform is None:
        transform = sitk.Transform()
        transform.SetIdentity()
    if output_pixel_type is None:
        output_pixel_type = image.GetPixelID()
    resample_filter = sitk.ResampleImageFilter()
    resample_filter.SetInterpolator(interpolator)
    resample_filter.SetTransform(transform)
    resample_filter.SetOutputPixelType(output_pixel_type)
    resample_filter.SetDefaultPixelValue(default_value)
    resample_filter.SetReferenceImage(reference_image)
    return resample_filter.Execute(image)

def generate_data_propotional(series_uid, scale, interpolator, output_pixel_type, is_dcm=True):
    if is_dcm:
        reader = sitk.ImageSeriesReader()
        filenames = reader.GetGDCMSeriesFileNames(series_uid)
        reader.SetFileNames(filenames)
        im = reader.Execute()
    else:
        im = sitk.ReadImage(series_uid)
    ori_spacing = im.GetSpacing()
    ori_size = im.GetSize()
    new_size = scale
#     for i in range(3):
#         new_size[i] = int(ori_spacing[i]*ori_size[i]/(ori_spacing[0]))
#   new_spacing = [ori_spacing[0]]*3
    
    new_spacing = np.empty(3)
    for i in range(3):
        new_spacing[i] = ori_size[i] * ori_spacing[i] / new_size[i]

#     interpolator = sitk.sitkNearestNeighbor

    black_im = sitk_new_blank_image(new_size, new_spacing, im.GetDirection(), im.GetOrigin())
    new_im = sitk_resample_to_image(im, black_im, interpolator, default_value=0, output_pixel_type=output_pixel_type)

    return new_im

In [3]:
src_root = '../data/brain_henan/bystudy'
all_data_file = '../data/brain_henan/algo_mask/brain_seg_by_threshold/out.csv'
mask_root = '../data/brain_henan/algo_mask/brain_seg_by_threshold'
outdir = '../data/brain_henan/algo_mask/brain_seg_by_threshold_preprocessed_512'
os.makedirs(outdir, exist_ok=True)
df_data_table = pd.read_csv(all_data_file)

substraction_data_list = []
mix_data_list = []
mask_list = []


for index, row in df_data_table.iterrows():
    mask_file = os.path.join(mask_root, row['剪影'])
    if not os.path.isdir(mask_file):
        continue
    mask_file = os.path.join(mask_file, 'mask_demo0.mha')
    if not os.path.isfile(mask_file):
        continue
    substraction_data_file = os.path.join(src_root, row['study_uid'], row['剪影'])
    if not os.path.isdir(substraction_data_file):
        continue
    mix_data_file = os.path.join(src_root, row['study_uid'], row['mix'])
    if not os.path.isdir(mix_data_file):
        continue
    mask_list.append(mask_file)
    substraction_data_list.append(substraction_data_file)
    mix_data_list.append(mix_data_file)

In [4]:
# infile = mix_data_list[0]

# rescale_size = [128, 128, 128]

# mix_data = generate_data_propotional(infile, rescale_size, sitk.sitkLinear, sitk.sitkInt16)
# substraction_data = generate_data_propotional(substraction_data_list[0], rescale_size, sitk.sitkLinear, sitk.sitkInt16)
# mask = generate_data_propotional(mask_list[0], rescale_size, sitk.sitkNearestNeighbor, sitk.sitkUInt8, False)


# print(mix_data.GetSize())
# print(substraction_data.GetSize())
# print(mask.GetSize())

In [5]:
rescale_size = [512, 512, 512]

for i in range(len(mask_list)):
    beg = time.time()
    print('====> begin to process {}'.format(mask_list[i]))
    mix_data = generate_data_propotional(mix_data_list[i], rescale_size, sitk.sitkLinear, sitk.sitkInt16)
    substraction_data = generate_data_propotional(substraction_data_list[i], rescale_size, sitk.sitkLinear, sitk.sitkInt16)
    mask = generate_data_propotional(mask_list[i], rescale_size, sitk.sitkNearestNeighbor, sitk.sitkUInt8, False)
    print('\ttime elpased when rescale data:{:.3f}'.format(time.time()-beg))
    
    
    # save to nii.gz
    basename = os.path.basename(substraction_data_list[i])
    out_name_mix = os.path.join(outdir, '{}_mix.nii.gz'.format(basename))
    out_name_substraction = os.path.join(outdir, '{}_substraction.nii.gz'.format(basename))
    out_name_mask = os.path.join(outdir, '{}_mask.nii.gz'.format(basename))
    
    sitk.WriteImage(mix_data, out_name_mix)
    sitk.WriteImage(substraction_data, out_name_substraction)
    sitk.WriteImage(mask, out_name_mask)
    print('\ttime elpased when save to nii:{:.3f}'.format(time.time()-beg))
    
    
#     # save to npy
#     out_name_mix_npy = os.path.join(outdir, '{}_mix.npy'.format(basename))
#     out_name_substraction_npy = os.path.join(outdir, '{}_substraction.npy'.format(basename))
#     out_name_mask_npy = os.path.join(outdir, '{}_mask.npy'.format(basename))
    
#     ori_mix_data = sitk.GetArrayFromImage(mix_data)
#     ori_substraction_data = sitk.GetArrayFromImage(substraction_data)
#     ori_mask = sitk.GetArrayFromImage(mask)
    
#     ori_mask[ori_mask == 7] = 1
    
#     with open(out_name_mix_npy, 'wb') as f:
#         np.save(f, ori_mix_data)
#     with open(out_name_substraction_npy, 'wb') as f:
#         np.save(f, ori_substraction_data)
#     with open(out_name_mask_npy, 'wb') as f:
#         np.save(f, ori_mask)
    
#     print('\ttime elpased when save npy:{:.3f}'.format(time.time()-beg))
            
#     # calculate min/max
#     mix_min = np.min(ori_mix_data)
#     mix_max = np.max(ori_mix_data)
    
#     substraction_min = np.min(ori_substraction_data)
#     substraction_max = np.max(ori_substraction_data)
    
#     print('\ttime elpased when calculate min&max:{:.3f}'.format(time.time()-beg))
    
#     # normalize volume data to [0,1] and save float to npy
#     out_name_mix_npy = os.path.join(outdir, '{}_mix_[0-1].npy'.format(basename))
#     out_name_substraction_npy = os.path.join(outdir, '{}_substraction_[0-1].npy'.format(basename))
    
#     ori_mix_data_0_1 = (ori_mix_data-mix_min)/(mix_max-mix_min)
#     ori_substraction_data_0_1 = (ori_substraction_data-substraction_min)/(substraction_max-substraction_min)
    
#     with open(out_name_mix_npy, 'wb') as f:
#         np.save(f, ori_mix_data_0_1)
#     with open(out_name_substraction_npy, 'wb') as f:
#         np.save(f, ori_substraction_data_0_1)
        
#     print('\ttime elpased when save normalize [0,1] npy:{:.3f}'.format(time.time()-beg))
    
    
#     # normalize volume data to [0,1] and save float to npy
#     out_name_mix_npy = os.path.join(outdir, '{}_mix_[-1-1].npy'.format(basename))
#     out_name_substraction_npy = os.path.join(outdir, '{}_substraction_[-1-1].npy'.format(basename))
    
#     tmp_v = mix_max+mix_min
#     ori_mix_data_0_1 = (ori_mix_data*2-tmp_v)/tmp_v
#     tmp_v = substraction_max+substraction_min
#     ori_substraction_data_0_1 = (ori_substraction_data*2-tmp_v)/tmp_v
    
#     with open(out_name_mix_npy, 'wb') as f:
#         np.save(f, ori_mix_data_0_1)
#     with open(out_name_substraction_npy, 'wb') as f:
#         np.save(f, ori_substraction_data_0_1)
        
#     print('\ttime elpased when save normalize [-1,1] npy:{:.3f}'.format(time.time()-beg))
    
    
    print('====> end to process {}, Time Elapsed:{:.3f}s\n'.format(mask_list[i], time.time()-beg))

====> begin to process ../data/brain_henan/algo_mask/brain_seg_by_threshold/1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240/mask_demo0.mha
	time elpased when rescale data:85.612
	time elpased when save to nii:139.783
====> end to process ../data/brain_henan/algo_mask/brain_seg_by_threshold/1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240/mask_demo0.mha, Time Elapsed:139.783s

====> begin to process ../data/brain_henan/algo_mask/brain_seg_by_threshold/1.3.12.2.1107.5.99.2.9594.30000018101817420531200000128/mask_demo0.mha
	time elpased when rescale data:84.573
	time elpased when save to nii:142.946
====> end to process ../data/brain_henan/algo_mask/brain_seg_by_threshold/1.3.12.2.1107.5.99.2.9594.30000018101817420531200000128/mask_demo0.mha, Time Elapsed:142.947s

====> begin to process ../data/brain_henan/algo_mask/brain_seg_by_threshold/1.3.12.2.1107.5.99.2.9594.30000016040823144126500004946/mask_demo0.mha
	time elpased when rescale data:82.942
	time elpased when save to

	time elpased when rescale data:162.652
	time elpased when save to nii:215.831
====> end to process ../data/brain_henan/algo_mask/brain_seg_by_threshold/1.3.12.2.1107.5.99.2.9594.30000019041407443398400000128/mask_demo0.mha, Time Elapsed:215.832s

